# 1. E1 validation error analysis before fine-tuning

Use saved predictions only; no model is loaded. This deliberately selected E1 gallery predates E2 fine-tuning and is not a frequency estimate. Inputs are the saved E0/E1 evaluations, split and original E1 error-analysis record. The original report gallery remains in `outputs/figures/E1_validation_error_gallery.png`; an optional new gallery writes separately.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import torch

# Set this explicitly on Kaggle to the extracted project folder.
PROJECT_ROOT = None
if PROJECT_ROOT is None:
    PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                         if (p / "src/models.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the extracted project containing src/ and data/processed/")
PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.project import RUNS, load_metadata, read_strings, find_image_directory, find_csv
DATA_ROOT = (Path("/kaggle/input/datasets/leocaoprojects/enge707-2026-s2")
             if Path("/kaggle").exists() else PROJECT_ROOT / "data/raw")
OUTPUT_ROOT = (Path("/kaggle/working/reproduction") if Path("/kaggle").exists()
               else PROJECT_ROOT / "outputs/reproduction")
TRAIN_IMAGE_DIRECTORY = None  # Optional existing cache or explicit train/train directory.
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
RUN_TRAINING = False
RUN_TEST_PREDICTION = False
RUN_EVALUATION = False
split, mapping = load_metadata(PROJECT_ROOT)
print("Project:", PROJECT_ROOT, "Device:", DEVICE)
print(split["split"].value_counts().to_dict())

In [ ]:
from src.analysis import select_e1_errors, error_gallery
RUN_GALLERY = False
predictions = read_strings(PROJECT_ROOT / "outputs/metrics" / RUNS["E1"] / "validation_predictions.csv")
validation = split.loc[split["split"].eq("validation")]
assert predictions["image_id"].tolist() == validation["image_id"].tolist()
assert predictions["true_class"].tolist() == validation["class"].tolist()
selected, pairs = select_e1_errors(predictions)
assert len(selected) == 12 and selected["image_id"].is_unique
assert selected["true_class"].ne(selected["predicted_class"]).all()
record = json.loads((PROJECT_ROOT / "outputs/metrics/E1_error_analysis_check.json").read_text())
assert selected["image_id"].tolist() == record["selected_ids"]
print("E1 errors:", int(pairs["count"].sum()), "of", len(validation))
display(pairs.head(10))
display(selected)
comparison = []
for experiment in ["E0", "E1"]:
    report = pd.read_csv(PROJECT_ROOT / "outputs/metrics" / RUNS[experiment] / "classification_report.csv")
    report = report.set_index("class")
    comparison.append(report["f1"].rename(experiment))
display(pd.concat(comparison, axis=1))

## 2. Selected gallery

Using seed 42, sort candidate IDs before sampling: select three errors from each of the three largest confusion pairs, then three from all remaining pairs. All 12 selected IDs belong to validation. Labels show true class, prediction and maximum softmax confidence; this is not calibrated certainty. No labels are declared incorrect and no unverified visual cause is assigned.

Six classes improved in E1 compared with E0; CatMouth and DogMouth F1 declined. The table above retains these class-level differences rather than relying only on overall accuracy.

In [ ]:
from IPython.display import Image, display
from matplotlib import pyplot as plt
display(Image(filename=str(PROJECT_ROOT / "outputs/figures/E1_validation_error_gallery.png"), width=1000))
if RUN_GALLERY:
    directory = TRAIN_IMAGE_DIRECTORY or find_image_directory(DATA_ROOT, split["image_id"])
    error_gallery(selected, directory, OUTPUT_ROOT / "E1_validation_error_gallery.png")
    plt.show()
else:
    print("Showing original gallery; no raw images reopened.")